# Área quemada y severidad con Sentinel-2 (dNBR)
**Producto de emergencia tipo "carta imagen" — ejemplo: incendio de Capilla del Monte (Córdoba), septiembre de 2024**

El notebook lee las imágenes **directamente de la nube** (catálogo STAC público de Sentinel-2 L2A en AWS, sin cuenta ni descargas pesadas) y procesa solo el área de interés.

**Flujo:**
1. Buscar escenas Sentinel-2 L2A pre y post incendio
2. Cargar B8A (NIR), B12 (SWIR) y SCL a 20 m, recortadas al área
3. Pasar a reflectancia y enmascarar nubes/sombras con SCL
4. Calcular NBR pre, NBR post y **dNBR = NBR_pre − NBR_post**
5. Clasificar severidad (umbrales Key & Benson, USGS)
6. Vectorizar el área quemada y calcular hectáreas
7. Exportar GeoTIFF, GeoPackage y CSV para armar la carta en QGIS

👉 Corré las celdas en orden (`Entorno de ejecución → Ejecutar todas`). Para otro incendio, cambiá solo la celda de **parámetros**.

## 1. Instalación

In [ ]:
!pip install -q pystac-client odc-stac rioxarray

In [ ]:
import os, zipfile
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import rioxarray  # habilita .rio en xarray
from rasterio.features import shapes
from shapely.geometry import shape
from pystac_client import Client
import odc.stac

odc.stac.configure_rio(cloud_defaults=True)
print("Librerías listas")

## 2. Parámetros
- `BBOX`: área de interés en grados (lon_min, lat_min, lon_max, lat_max). Podés sacarla de QGIS o de [bboxfinder.com](http://bboxfinder.com).
- Fechas: usá [NASA FIRMS](https://firms.modaps.eosdis.nasa.gov/map/) para ver cuándo hubo focos. La ventana **pre** termina antes del inicio; la **post** empieza apenas se extingue (antes de lluvias y rebrote).

In [ ]:
NOMBRE_EVENTO = "Incendio Capilla del Monte - sep 2024"

BBOX = (-64.65, -31.00, -64.35, -30.70)   # lon_min, lat_min, lon_max, lat_max

FECHAS_PRE  = "2024-08-15/2024-09-08"
FECHAS_POST = "2024-09-25/2024-10-20"

MAX_NUBES = 30      # % de nubosidad máxima de la escena para buscar
UMBRAL    = 0.27    # dNBR mínimo para considerar "quemado"
MIN_HA    = 5       # superficie mínima de un polígono quemado (ha)
SALIDA    = "salida_dnbr"

## 3. Buscar escenas disponibles

In [ ]:
catalogo = Client.open("https://earth-search.aws.element84.com/v1")

def buscar(rango):
    items = catalogo.search(
        collections=["sentinel-2-l2a"], bbox=BBOX, datetime=rango,
        query={"eo:cloud_cover": {"lt": MAX_NUBES}},
    ).item_collection()
    items = sorted(items, key=lambda it: it.properties["eo:cloud_cover"])
    tabla = pd.DataFrame([{
        "idx": i,
        "fecha": it.datetime.strftime("%Y-%m-%d"),
        "nubes_%": round(it.properties["eo:cloud_cover"], 1),
        "tile": it.properties.get("s2:mgrs_tile", it.properties.get("grid:code", "")),
        "id": it.id,
    } for i, it in enumerate(items)])
    return items, tabla

items_pre, tabla_pre = buscar(FECHAS_PRE)
items_post, tabla_post = buscar(FECHAS_POST)
print("Escenas PRE (ordenadas por nubosidad):");  display(tabla_pre)
print("Escenas POST (ordenadas por nubosidad):"); display(tabla_post)

## 4. Elegir y cargar las escenas
Por defecto toma la menos nubosa de cada ventana (`idx = 0`). Si la vista previa de la sección 6 muestra nubes sobre el incendio, probá con otro `idx` de la tabla.

In [ ]:
IDX_PRE, IDX_POST = 0, 0

# Proyección UTM según el centro del área (misma grilla para ambas fechas)
lon_c, lat_c = (BBOX[0] + BBOX[2]) / 2, (BBOX[1] + BBOX[3]) / 2
EPSG = (32700 if lat_c < 0 else 32600) + int((lon_c + 180) // 6) + 1

def offset_de(item):
    # Desde 2022 (baseline >= 04.00) L2A trae un offset de 1000 que hay que restar,
    # salvo que el catálogo ya lo haya corregido.
    p = item.properties
    if p.get("earthsearch:boa_offset_applied", False):
        return 0
    return 1000 if float(p.get("s2:processing_baseline", "0")) >= 4.0 else 0

def cargar(item):
    ds = odc.stac.load(
        [item], bands=["nir08", "swir22", "scl", "red"],   # nir08 = B8A, swir22 = B12
        bbox=BBOX, crs=f"EPSG:{EPSG}", resolution=20,
    ).isel(time=0)
    return ds, offset_de(item)

item_pre, item_post = items_pre[IDX_PRE], items_post[IDX_POST]
ds_pre,  off_pre  = cargar(item_pre)
ds_post, off_post = cargar(item_post)
FECHA_PRE  = item_pre.datetime.strftime("%Y-%m-%d")
FECHA_POST = item_post.datetime.strftime("%Y-%m-%d")
print(f"PRE : {item_pre.id}  (offset {off_pre})")
print(f"POST: {item_post.id}  (offset {off_post})")
print(f"Grilla: {ds_pre.sizes['y']} x {ds_pre.sizes['x']} píxeles de 20 m, EPSG:{EPSG}")

## 5. NBR, dNBR y severidad

In [ ]:
# Clases SCL descartadas: 0 sin dato, 1 saturado, 3 sombra de nube, 8-9 nube, 10 cirros, 11 nieve.
# La clase 2 (dark area) se conserva: las cicatrices recientes suelen caer ahí.
SCL_INVALIDAS = [0, 1, 3, 8, 9, 10, 11]

CLASES = [  # código, mín, máx, etiqueta, color
    (1, -np.inf, -0.10, "Rebrote / aumento de vegetación", "#1a9850"),
    (2, -0.10,  0.10,  "No quemado",              "#d9d9d9"),
    (3,  0.10,  0.27,  "Severidad baja",          "#fee08b"),
    (4,  0.27,  0.44,  "Severidad moderada-baja", "#fdae61"),
    (5,  0.44,  0.66,  "Severidad moderada-alta", "#f46d43"),
    (6,  0.66,  np.inf, "Severidad alta",         "#a50026"),
]

def reflectancia(da, offset):
    v = da.values.astype("float32")
    v[v == 0] = np.nan                     # 0 = sin dato en L2A
    return (v - offset) / 10000.0

def calcular_nbr(ds, offset):
    nir, swir = reflectancia(ds["nir08"], offset), reflectancia(ds["swir22"], offset)
    with np.errstate(divide="ignore", invalid="ignore"):
        nbr = (nir - swir) / (nir + swir)
    nbr[np.isin(ds["scl"].values, SCL_INVALIDAS)] = np.nan
    return nbr

nbr_pre  = calcular_nbr(ds_pre,  off_pre)
nbr_post = calcular_nbr(ds_post, off_post)
dnbr = nbr_pre - nbr_post

severidad = np.zeros(dnbr.shape, dtype="uint8")     # 0 = sin dato
for cod, mn, mx, _, _ in CLASES:
    severidad[(dnbr >= mn) & (dnbr < mx)] = cod

transform = ds_pre.rio.transform()
px_ha = abs(transform.a * transform.e) / 10000.0
tabla_sev = pd.DataFrame([{
    "clase": et, "hectareas": round(float((severidad == cod).sum()) * px_ha, 1)
} for cod, _, _, et, _ in CLASES])

print(f"Celdas válidas en ambas fechas: {np.isfinite(dnbr).mean()*100:.1f} %")
display(tabla_sev)

## 6. Vista previa

In [ ]:
def falso_color(ds, offset):
    # SWIR-NIR-Rojo: lo quemado se ve marrón rojizo, la vegetación verde
    rgb = np.dstack([reflectancia(ds[b], offset) for b in ("swir22", "nir08", "red")])
    lo, hi = np.nanpercentile(rgb, 2), np.nanpercentile(rgb, 98)
    return np.clip((rgb - lo) / (hi - lo), 0, 1)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(falso_color(ds_pre, off_pre));   ax[0].set_title(f"Pre ({FECHA_PRE}) B12-B8A-B4")
ax[1].imshow(falso_color(ds_post, off_post)); ax[1].set_title(f"Post ({FECHA_POST}) B12-B8A-B4")

cmap = ListedColormap(["#ffffff00"] + [c for *_, c in CLASES])
norm = BoundaryNorm(np.arange(-0.5, 7.5), cmap.N)
ax[2].imshow(severidad, cmap=cmap, norm=norm, interpolation="nearest")
ax[2].set_title("Severidad (dNBR)")
ax[2].legend(handles=[Patch(color=c, label=e) for _, _, _, e, c in CLASES],
             loc="lower left", fontsize=8, framealpha=0.9)
for a in ax: a.axis("off")
plt.tight_layout(); plt.suptitle(NOMBRE_EVENTO, fontsize=14, y=1.03); plt.show()

## 7. Vectorizar el área quemada

In [ ]:
quemado = (np.nan_to_num(dnbr, nan=-9) >= UMBRAL).astype("uint8")
polis = [shape(g) for g, v in shapes(quemado, mask=quemado == 1, transform=transform) if v == 1]
gdf = gpd.GeoDataFrame(geometry=polis, crs=f"EPSG:{EPSG}")
if not gdf.empty:
    gdf["ha"] = (gdf.area / 10000).round(1)
    gdf = gdf[gdf["ha"] >= MIN_HA].reset_index(drop=True)
    gdf["umbral_dnbr"] = UMBRAL
    gdf["fecha_pre"], gdf["fecha_post"] = FECHA_PRE, FECHA_POST

print(f"Área quemada (dNBR ≥ {UMBRAL}, polígonos ≥ {MIN_HA} ha): "
      f"{gdf['ha'].sum() if not gdf.empty else 0:,.1f} ha en {len(gdf)} polígonos")
if not gdf.empty:
    display(gdf.sort_values("ha", ascending=False).head(10).drop(columns="geometry"))

## 8. Exportar para QGIS
Genera un `.zip` con:
- `dnbr.tif` y `severidad.tif` (con `severidad.qml`: QGIS la abre ya coloreada)
- `falso_color_post.tif` para usar de fondo en la carta
- `area_quemada.gpkg` y `superficie_por_clase.csv`

In [ ]:
os.makedirs(SALIDA, exist_ok=True)
plantilla = ds_pre["nir08"]

def guardar(arr, nombre, dtype, nodata):
    da = plantilla.copy(data=arr.astype(dtype)).rio.write_nodata(nodata)
    da.rio.to_raster(os.path.join(SALIDA, nombre), compress="deflate")

guardar(dnbr, "dnbr.tif", "float32", np.nan)
guardar(severidad, "severidad.tif", "uint8", 0)

fc = (falso_color(ds_post, off_post) * 255).astype("uint8")
plantilla.expand_dims(band=3).copy(data=np.moveaxis(fc, -1, 0)) \
    .rio.to_raster(os.path.join(SALIDA, "falso_color_post.tif"), compress="deflate")

entradas = "\n".join(f'        <paletteEntry value="{c}" color="{col}" label="{e}" alpha="255"/>'
                     for c, _, _, e, col in CLASES)
with open(os.path.join(SALIDA, "severidad.qml"), "w", encoding="utf-8") as f:
    f.write(f'''<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.28.0"><pipe>
  <rasterrenderer type="paletted" band="1" opacity="1" alphaBand="-1" nodataColor="">
    <colorPalette>
{entradas}
    </colorPalette>
  </rasterrenderer>
</pipe></qgis>''')

tabla_sev.to_csv(os.path.join(SALIDA, "superficie_por_clase.csv"), index=False)
if not gdf.empty:
    gdf.to_file(os.path.join(SALIDA, "area_quemada.gpkg"), layer="area_quemada", driver="GPKG")

zip_path = f"{SALIDA}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(SALIDA):
        z.write(os.path.join(SALIDA, f), arcname=f)
print("Archivos:", sorted(os.listdir(SALIDA)))

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Listo: {zip_path}")

## 9. Notas para la carta imagen y la entrevista
- **Fuente:** Sentinel-2 L2A (Copernicus/ESA), vía Earth Search (AWS). Indicar fechas pre y post.
- **Método:** dNBR = NBR_pre − NBR_post con NBR = (B8A − B12)/(B8A + B12); umbrales de Key & Benson (USGS).
- **Leyenda obligatoria:** *Producto preliminar, sin validación de campo.*
- **Limitaciones:** umbral fijo (conviene calibrarlo localmente), sombras topográficas en sierras, falsos positivos por cambios en lotes agrícolas, nubes/humo en la fecha post.
- **Próximo paso:** con humo o nubes el óptico no sirve; ahí entra el radar (SAOCOM, banda L).